In [ ]:
from sklearn.mixture import GaussianMixture
%load_ext autoreload
%autoreload 2
import logging
from src.exp.gen.generate import NoiseType, IvType, FunType, DagType, IvMode, gen_data_type
from src.mixtures.mixing.mixing import MixingType
from src.examples.util import demo_clustering
from src.exp.algos import CD
from src.exp.gen.generate import GSType

### Fig. 1: Mixing for a causal relationship X -> Y

In [ ]:
seed = 42
params = { 'N': 2, 'S': 1000, 'P': 1, 'K': 2, 'C': 5,  'PZ': 0.5, 'NZ': 2,
           'NS': NoiseType.GAUSS, 'F': FunType.LIN, 'DG': DagType.ERDOS,
           'IVT': IvType.FLIP, 'IVM': IvMode.MIXING,
           'GS': GSType.BIV_CAUSAL_CHANGEY}

data, truths = gen_data_type(params, seed)
print(f"\tMixed Nodes: {truths['t_n_Z']}" )
truths["_dg"].plot_X(data)

In [ ]:
# Discover the mixture
(ours, _) = demo_clustering(data, truths, params, mixing_ty=MixingType.MIX_LIN, causaldiscovery_method_ty = CD.SKIP,  ORACLE_G = True, ORACLE_K = False, KMAX=5, ret_model=True)
#truths["_dg"].plot_X_idls(data,e_Z_n)
print(f"\tDiscovered Nodes: {ours.e_n_Z}" )


In [ ]:
import pandas as pd
from src.mixtures.util.utils_idl import pi_join
import numpy as np

i = 2
pa_i = list(truths["_dg"].G.predecessors(i))

true_labels = np.zeros(data.shape[0])
for zi, node_set in enumerate(truths["_dg"].conf_ind_sets):
    if i in node_set:
        true_labels = pi_join(true_labels, truths["_dg"].Zs[zi])

for ix, pa in enumerate(pa_i):
    df = pd.DataFrame({
        'x': data[:, pa],
        'y': data[:, i],
        'c':  true_labels
    })

    df.to_csv('illustration_1.tsv', sep='\t', index=False)
    df = pd.read_csv('../../results_paper/illustration_1.tsv', sep='\t')

    df['x'] = (df['x'] - df['x'].min()) / (df['x'].max() - df['x'].min())
    df['y'] = (df['y'] - df['y'].min()) / (df['y'].max() - df['y'].min())
    df.to_csv('illustration_1_nm.tsv', sep='\t', index=False)


In [ ]:
truths["_dg"].plot_X_idls(data,ours.e_Z_n)

### Synthetic data generation and CMM fitting as in Fig. 4

In [ ]:
# Synthetic data parameters
params = {
    'N': 10, 'S': 1000, 'P': 1, 'K': 2, 'C': 5, 'PZ': 0.5, 'NZ': 2,
    'NS': NoiseType.GAUSS, 'F': FunType.LIN, 'DG': DagType.ERDOS,
    'IVT': IvType.FLIP, 'IVM': IvMode.MIXING, 'GS': GSType.GRAPH }

data, truths = gen_data_type(params, 42)
print(f"\tMixed Nodes: {truths['t_n_Z']}")
#truths["_dg"].plot_X(data)

# Discover the mixture
KMAX = 5
(ours, _) = demo_clustering(data, truths, params, mixing_ty=MixingType.MIX_LIN, causaldiscovery_method_ty=CD.SKIP,  ORACLE_G=True, ORACLE_K=False, KMAX=KMAX, ret_model=True)
#truths["_dg"].plot_X_idls(data,e_Z_n)
print(f"\tDiscovered Nodes: {ours.e_n_Z}" )

In [ ]:
for i in ours.topic_graph.nodes:
    ours.visu_pproba_dens(i)


In [ ]:
ours.visu_heatmatrix_nodepair_MI(hide_singleclus=True)